In [1]:
# CÉLULA 1: Carregando e preparando os dados

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Lendo o arquivvo CSV
tabela_banco = pd.read_csv('historico_credito_1000_clientes.csv')

print("Visualizando os primeiros 5 clientes da base de dados:\n")
display(tabela_banco.head())

# 2. Separando as variáveis de entrada (X) e variaável alvo (y)
# X = informações usadas para tomar a decisão
# y = resposta que o modelo deve aprender a prever
X = tabela_banco[['Idade','Salario','Divida_Atual']]
y = tabela_banco['Risco_Alto']

print("\nQuantidade de clientes na base:", len(tabela_banco))
print("\nDistribuição da variável alvo:")
print(y.value_counts())

FileNotFoundError: [Errno 2] No such file or directory: 'historico_credito_1000_clientes.csv'

In [ ]:
# CÉLULA 2: Separando treino e teste

# Separando 80% dos dados para treino e 20% para teste
# stratify=y mantém a proporção de risco alto e baixo nos dois conjuntos

X_treino, X_teste, y_treino, y_teste = \
train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Tamanho do conjunto doe treino:", X_treino.shape)
print("Tamanho do conjunto de teste:", X_teste.shape)

In [ ]:
# CÉLULA 3: Normalizando os dados

# Redes Neurais trabalham melhor quando os dados estão em escalas parecidas
# Exemplo:
# Idade pode variar de 18 a 80
# Salário pode variar de 1000 a 20000
# Dívida pode variar de 0 a 50000
# Se não normalizarmos, valores grandes como salário e dívida atual
# podem dominar o treinamento

normalizador = StandardScaler()

# O normalizado aprende média e desvio padrão
# apenas com os dados de treino

X_treino_normalizado = normalizador.fit_transform(X_treino)


# O teste é apenas transformado usando a mesma regra aprendida no treino
X_teste_normalizado = normalizador.transform(X_teste)

print("Dados normalizados com sucesso!")

In [ ]:
# CÉLULA 4: Construindo a Rede Neural Artificial

import tensorflow as tf
from tensorflow import keras

tf.random.set_seed(42)

# Criando a arquitetura da rede neural
modelo_rn = keras.Sequential([

    # Camada de entrada:
    # Temos 3 informações de entrada: Idade, Salario, Divida_Atual
    keras.Input(shape=(3,)),

    # Primeira camada oculta:
    # 8 neurônios tentando encontrar padrões nos dados
    keras.layers.Dense(8, activation='relu'),

    # Segunda camada oculta:
    # 4 neurônios refinando os padrões encontrados
    keras.layers.Dense(4, activation='relu'),

    # Camada de saída:
    # 1 neurônio porque queremos prever apenas uma resposta:
    # risco alto ou não
    # sigmoid gera uma probabilidade entre 0 e 1
    keras.layers.Dense(1, activation='sigmoid')
])

# Compilando o modelo
modelo_rn.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("Rede Nural construída com sucesso!\n")
modelo_rn.summary()

In [ ]:
# CÉLULA 5: Treinando a Rede Neural

print("Iniciando o treinamento da rede Neural ...\n")

historico = modelo_rn.fit(
    X_treino_normalizado,
    y_treino,
    epochs=100,
    validation_split=0.2,
    verbose=1
)

print("Treinamento concluído!")

In [ ]:
# CÉLULA 6: Avaliando o modelo com dados de teste

perda, acuracia = modelo_rn.evaluate(X_teste_normalizado, y_teste)

print(f"\nPerda no teste: {perda:.4f}")
print(f"Acuracia na prova final: {acuracia * 100:.2f}%")

In [ ]:
# CÉLULA 7: Simulando a análise de novo cliente

novo_cliente = {
    'Idade': [20],
    'Salario': [3000],
    'Divida_Atual': [2000]
}

novo_cliente_df = pd.DataFrame(novo_cliente)

# O novo cliente precisa passar pela mesma normalização usada no treino
novo_cliente_normalizado = normalizador.transform(novo_cliente_df)

# A rede neural calcula uma probabilidade entre 0 e 1
predicao = modelo_rn.predict(novo_cliente_normalizado)

probabilidade_risco = predicao[0][0]

print(
    f"Probabilidade de inadimplência: {probabilidade_risco * 100:.2f}%\n"
)

# Regra de negócio:
# Se a probabilidade for maior que 50%, considera risco alto
if probabilidade_risco > 0.5:
    print("CRÉDITO REPROVADO: Alto risco na operação.")
else:
    print("CRÉDITO APROVADO: Dentro da margem de segurança.")